In [1]:
getwd()

[1] "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/scripts"

In [1]:
setwd("/mnt/gsdata/projects/panops/panops-data-registry/data/flux")

In [2]:
packages <- c("bigleaf", "purrr", "REddyProc", "stringr", "tidyr", "dplyr", "plyr", "broom", "lutz", "sf")

# Install packages not yet installed
installed_packages <- packages %in% rownames(installed.packages())
if (any(installed_packages == FALSE)) {
  install.packages(packages[!installed_packages])
}

invisible(lapply(packages, library, character.only = TRUE))



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


------------------------------------------------------------------------------

You have loaded plyr after dplyr - this is likely to cause problems.
If you need functions from both plyr and dplyr, please load plyr first, then dplyr:
library(plyr); library(dplyr)

------------------------------------------------------------------------------


Attaching package: ‘plyr’


The following objects are masked from ‘package:dplyr’:

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


The following object is masked from ‘package:purrr’:

    compact


Linking to GEOS 3.12.2, GDAL 3.9.2, PROJ 9.5.0; sf_use_s2() is TRUE



In [3]:
# ++ Function to calculate time zone

TimeZoneCalculatorFLUXNET <- function(lat = lat, long = long) {

  print('Calculation of Time zone')

  aaa <- tz_lookup_coords(lat, lon, method = "accurate", warn = FALSE)
 
  dfout <- tz_offset(as.POSIXct("2018-01-01 12:00:00", tz = tz_lookup_coords(-0.000500, 51.476852)), aaa)
  
  TimeZone_h.n<-dfout$utc_offset_h  
  return(TimeZone_h.n)
  
}


In [4]:
# ++ Function to calculate the functional properties

EFPcalc<-function(dat.all){

  #-- Set the growing season filter as 30% of the seasonal amplitude of daily GPP
  
  GSfilt <- 0.3

  if(is.na(summary(dat.all$P)[4]) | (summary(dat.all$P)[4] == 0)){
    print("##### Warning: no Precipitation data!!! No precipitation Filter")
    dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                                GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                                vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
    precipAvail <- "no"
  } 
  
  if(!is.na(summary(dat.all$P)[4]) & (summary(dat.all$P)[4] != 0)){
    print("#### Precipitation data available!!! Precipitation Filter Active")
    # -- Indicate if the data are hourly (record.per.hour <- 1) or half-hourly (record.per.hour <- 2)
    record.per.hour <- 2
    
    dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=TRUE, 
                                GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                precip="P", tprecip=0.1,precip.hours=24,records.per.hour=record.per.hour,
                                vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
    precipAvail <- "yes"
  }  
  
  # Filtering G1 and WUE, remove also low u* values (u* <= 0.2)
  
  print('....computing WUE Metrics for the site')
  
  # Water-use efficiency (WUE):
  #   WUE = GPP / ET
  # 
  # Water-use efficiency based on NEE (WUE_NEE):
  #   WUE_NEE = NEE / ET
  #
  # Inherent water-use efficiency (IWUE; Beer et al. 2009):
  #   IWUE = (GPP * VPD) / ET
  # 
  # Underlying water-use efficiency (uWUE; Zhou et al. 2014):
  #   uWUE= (GPP * sqrt(VPD)) / ET
  # 
  # All metrics are calculated based on the median of all values. E.g. WUE = median(GPP/ET,na.rm=TRUE)
  # 
  aaa<-subset(dat.filtered, SW_IN > 200 & USTAR > 0.2)
  
  EFPsOut<-WUE.metrics(aaa, GPP = "GPP_NT", NEE = "NEE", LE = "LE", VPD = "VPD_kPa",
                       Tair = "TA", constants = bigleaf.constants())
  
  # Calculate maximum Evapotranspiration (95th percetile of half hourly values)
  
  print('....computing maximum Evapotranspiration')
  
  EFPsOut$ETmax<-quantile(aaa$ET, 0.95, na.rm=T)

  # Availability of precipitation
  EFPsOut$precipAvail <- precipAvail
  
  # Calculation of surface conductance and stomatal slope, G1
  print('....computing maximum surface conductance (Gsmax) and stomatal slope (G1)')
  
  # Using only measured data
  # -- Removing NA for wind speed and calculating air pressure from elevation data (Elevation is needed)
  
  aaa<- aaa %>% drop_na(WS)
  
  # -- Calculation of aerodynamic conductance
  
  if (dim(aaa)[1] != 0){
    aaa$pressure<-pressure.from.elevation(elevation,aaa$TA,aaa$VPD_kPa)
    Ga <- aerodynamic.conductance(aaa,Tair = "TA", pressure = "pressure",
                                  wind = "WS", ustar = "USTAR", H = "H", 
                                  Rb_model="Thom_1972")[,"Ga_h"]
  }
  
  if (dim(aaa)[1] == 0){
    aaa<-subset(dat.filtered, SW_IN > 200 & USTAR > 0.2)
    aaa$pressure<-pressure.from.elevation(elevation,aaa$TA,aaa$VPD_kPa)
    Ga <- 1
  }
  
  if(all(is.na(aaa$G))){
    Gs <- surface.conductance(aaa,Ga=Ga, Tair = "TA", pressure = "pressure", Rn = "NETRAD",
                              G = NULL, S = NULL, VPD = "VPD_kPa", LE = "LE",  
                              missing.G.as.NA = FALSE, 
                              missing.S.as.NA = FALSE)
    EFPsOut$Gavail<-"no"
  }
  
  if(!all(is.na(aaa$G))){
    Gs <- surface.conductance(aaa,Ga=Ga, Tair = "TA", pressure = "pressure", Rn = "NETRAD",
                              G = "G", S = NULL, VPD = "VPD_kPa", LE = "LE",  
                              missing.G.as.NA = FALSE, 
                              missing.S.as.NA = FALSE)
    EFPsOut$Gavail<-"yes"
  }
  
  aaa$Gs_mol<-Gs$Gs_mol
  
  aaa<- aaa %>% drop_na(Gs_mol)
  aaa <- subset(aaa, VPD_kPa > 0)

  #-- Calculation of maximum surface conductance as the 90th percentile of half hourly Gs
  EFPsOut$GSmax<-quantile(na.omit(Gs$Gs_ms), 0.90)
  
  nmin <- 40 #Set as 40 the minimum number of good data points to calculate the functions
  
  if(dim(aaa)[1] >= nmin){  
    
    if(all(is.na(aaa$CO2))){
      #- Fix CO2 concentration to 400 ppm if not available
      aaa$CO2 <- 400
      ### Use robust regression to minimize influence of outliers in Gs                           
      mod_USO <- stomatal.slope(aaa,model="USO",GPP="GPP_NT",Gs="Gs_mol", Tair = "TA", pressure = "pressure", 
                                VPD = "VPD_kPa", Ca = "CO2",robust.nls=TRUE,nmin=nmin,fitg0=FALSE)
      
      EFPsOut$CO2avail<-"no"
      
    }
    
    if(!all(is.na(aaa$CO2))){
      
      ### Use robust regression to minimize influence of outliers in Gs                           
      mod_USO <- stomatal.slope(aaa,model="USO",GPP="GPP_NT",Gs="Gs_mol", Tair = "TA", pressure = "pressure", 
                                VPD = "VPD_kPa", Ca = "CO2",robust.nls=TRUE,nmin=nmin,fitg0=FALSE)
      
      EFPsOut$CO2avail<-"yes"
      
    }
    
    EFPsOut$g1<-tidy(mod_USO)$estimate
    EFPsOut$g1_stderr<-tidy(mod_USO)$std.error
  }
  
  if(dim(aaa)[1] < nmin){  
    EFPsOut$g1<-NA
    EFPsOut$g1_stderr<-NA
  }

  print('....computing Evaporative fraction for the sites and its amplitude')
  
  aaa$EF <- aaa$LE/(aaa$LE+aaa$H)
  EFPsOut$EF <- median(aaa$EF, na.rm = TRUE)
  EFPsOut$EFampl <- quantile(na.omit(aaa$EF), 0.75, na.rm = TRUE) - quantile(na.omit(aaa$EF), 0.25, na.rm = TRUE)
  
  ### -- print('....computing LRC parameters for the site')
  print('....computing LRC parameters for the site')
  
  myLRC<-function(datafilt){
    
    numNAN<-150
    #if(sum(is.na(datafilt$NEE)) >= numNAN) return(NA)
    #browser()
    if(sum(is.na(datafilt$NEE))/length(datafilt$NEE) >= 0.8) return(NA)
    
    if(sum(is.na(datafilt$NEE))/length(datafilt$NEE) < 0.8){
      #print(paste("...optimizing..", unique(datafilt$FiveDaySeq)))
      # browser()
      result = tryCatch({
        fitLRC<-light.response(datafilt, NEE = "NEE", Reco = "Reco", PPFD = "PPFD",
                               PPFD_ref = 2000)
        out<-tidy(fitLRC)$estimate[2]
        return(out)
      }, error = function(e) {
        return(NA)
        
      })
      
      
    }
  }
  
  #Filter again data out all data with QC > 1 (retaining good quality 0 and 1), growing season, no precip filter
  
  dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                              GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                              precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                              vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
  
  
  dat.filtered$FiveDaySeq<-rep(c(1:ceiling(dim(dat.filtered)[1]/5)),each = 48*5, length.out=dim(dat.filtered)[1])
  
  zzz<-data.frame(NEE=dat.filtered$NEE,
                  PPFD=dat.filtered$PPFD_IN_FROM_SWIN,
                  Reco=dat.filtered$RECO_NT,
                  FiveDaySeq = dat.filtered$FiveDaySeq)
  
  outGPPsat <- unlist(by(zzz, zzz$FiveDaySeq, myLRC), use.names=FALSE)
  
  EFPsOut$GPPsat<-quantile(outGPPsat, 0.90, na.rm = T)  
  EFPsOut$NEPmax<-quantile(na.omit(subset(zzz, PPFD > 200*2.11)$NEE*-1), 0.99)
  
  ### -- print('....computing Basal respiration with REddyProc')
  print('....computing Rb parameters for the site')
  
  ttt<-data.frame(dat.all)
  #ttt$DateTime<-ttt$DateTime + 30*30
  TimeZone <- TimeZoneCalculatorFLUXNET(lat, lon)
  
  EddyProc.C <- sEddyProc$new(site, ttt, c('NEE', 'NEE_QC_OK', 'SW_IN', 'SW_IN_QC_OK',
                                           'TA','TA_QC_OK'),LatDeg=as.numeric(lat), LongDeg=as.numeric(lon),TimeZoneHour=TimeZone)
  
  #-- Run the nighttime partitioning for the calculation  of basal respiration
  
  print('Run the nighttime partitioning for the calculation  of basal respiration')
  
  EddyProc.C$sMRFluxPartition(FluxVar.s = "NEE", QFFluxVar.s = "NEE_QC_OK", 
                              QFFluxValue.n = 1, TempVar.s = "TA", QFTempVar.s = "TA_QC_OK", 
                              QFTempValue.n = 1, RadVar.s = "SW_IN", T_ref.n = 273.15 + 15, 
                              Suffix.s = "")
  
  EFPsOut$Rb<-mean(EddyProc.C$sTEMP$R_ref, na.rm = T)
  EFPsOut$Rbmax<-quantile(EddyProc.C$sTEMP$R_ref, 0.95)

  #-- Apparent Carbon Use Efficiency
  print('Calculation of apparent Carbon Use Efficiency, aCUE')
  
  dat.all$Rb <- ifelse(is.null(EddyProc.C$sTEMP$R_ref), NA, EddyProc.C$sTEMP$R_ref)
  
  dat.filtered.CUE <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                                  GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                  precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                                  vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)

  daily.aggr<-ddply(dat.filtered.CUE, .(year, doy), summarize, mean_GPP=mean(GPP_NT), mean_Rb=mean(Rb))
  
  EFPsOut$aCUE <- median(1-(daily.aggr$mean_Rb/daily.aggr$mean_GPP), na.rm=TRUE)
  EFPsOut$TZ<-as.numeric(TimeZone)
  EFPsOut$nyears<-nyears
  EFPsOut$SITE_ID <- site
  
  EFPsF15LT<-unlist(EFPsOut)
  
  names(EFPsF15LT) <- str_replace_all(names(EFPsF15LT),c("ETmax.95%" = "ETmax",
                                     "GSmax.90%" = "GSmax",
                                     "g1" = "G1",
                                     "EFampl.75%" = "EFampl",  
                                     "GPPsat.90%" = "GPPsat",
                                     "NEPmax.99%" = "NEPmax",
                                     "Rbmax.95%" = "Rbmax"))
  
  EFPsF15LT <- data.frame(t(EFPsF15LT))
  output.EFPs <- subset(EFPsF15LT, select = c("uWUE","ETmax","precipAvail","Gavail","GSmax","CO2avail","G1",
                              "EF","EFampl","GPPsat","NEPmax","Rb","Rbmax","aCUE","TZ","nyears","SITE_ID"))
  
  print(paste0("...Finish site ",site))
  
  return(output.EFPs)
}

In [6]:
ClimateCalc <- function(dat.all){
  # ---- Climate Descriptors ----
  print("....computing climate descriptors")
  
  out <- list()
  
  # Mean annual precipitation (mm)
  out$P_mean <- mean(dat.all$P, na.rm = TRUE)
  
  # Growing season filtered data: use same GPP filter as EFPcalc
  GSfilt <- 0.3
  gs_data <- filter.data(
    data.frame(dat.all),
    quality.control = TRUE,
    filter.growseas = TRUE,
    filter.precip = FALSE,
    GPP = "GPP_NT", doy = "doy", year = "year", tGPP = GSfilt,
    precip = "P", tprecip = 0.1, precip.hours = 24, records.per.hour = 2,
    vars.qc = c("TA","H","LE","NEE","VPD"), quality.ext = "_QC", good.quality = 1
  )
  
  # Means during growing season
  out$VPD_mean   <- mean(gs_data$VPD,   na.rm = TRUE)   # hPa
  out$SWin_mean  <- mean(gs_data$SW_IN, na.rm = TRUE)   # W/m2
  out$Tair_mean  <- mean(gs_data$TA,    na.rm = TRUE)   # °C
  
  # Cumulative Soil Water Index (if available)
  if("CSWI" %in% names(dat.all)){
    out$CSWI_cum <- sum(dat.all$CSWI, na.rm = TRUE)
  } else {
    out$CSWI_cum <- NA
  }
  
  return(data.frame(t(unlist(out))))
}

From here we can use the functions and run the code based on our available data

In [5]:
library(data.table)# Create a safe version of EFPcalc to avoid breaking the loop
tower_meta <- fread("combined_site_metadata.csv")


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [6]:
tower_flux_folder <- "merged_sites_subset_renamed"

In [7]:
tower_flux_files <- list.files(
  path = tower_flux_folder,
  pattern = "\\.csv$",
  full.names = TRUE
)

file_lookup <- setNames(
  as.list(tower_flux_files),
  stringr::str_remove(
    tools::file_path_sans_ext(basename(tower_flux_files)),
    "_merged$"
  )
)


In [8]:
safeEFPcalc <- purrr::possibly(EFPcalc, otherwise = NA)

In [9]:
efp_results_list <- list()

for (this_site in unique(tower_meta$SITE_ID)) {
  message("Processing site: ", this_site)

  site_info <- tower_meta %>% filter(SITE_ID == this_site)
  if (any(is.na(site_info$LOCATION_ELEV), is.na(site_info$LOCATION_LAT), is.na(site_info$LOCATION_LONG))) {
    warning("Missing location info for site ", this_site, "; skipping.")
    next
  }

  flux_path <- file_lookup[[this_site]]
if (is.null(flux_path) || !file.exists(flux_path)) {
  warning("No merged flux file found for site ", this_site, "; skipping.")
  next
}

  flux_df <- data.table::fread(flux_path) %>%
    mutate(
      DateTime = lubridate::ymd_hm(as.character(TIMESTAMP_START), tz = "UTC") +
        lubridate::minutes(15),
      year  = lubridate::year(DateTime),
      doy   = lubridate::yday(DateTime),
      ET    = LE.to.ET(LE, TA) * 1800,
      precip_mm          = P,
      VPD_kPa            = VPD / 10,
      PPFD_IN_FROM_SWIN  = SW_IN * 2.11,
      NEE_QC_OK = NEE_QC,
      SW_IN_QC_OK = SW_IN_QC,
      TA_QC_OK = TA_QC,
      SITE_ID = this_site
    )

  if (nrow(flux_df) == 0) {
    warning("No data for site ", this_site, "; skipping.")
    next
  }

  site      <- this_site
  elevation <- as.numeric(site_info$LOCATION_ELEV)
  lat       <- as.numeric(site_info$LOCATION_LAT)
  lon       <- as.numeric(site_info$LOCATION_LONG)
  nyears    <- dplyr::n_distinct(flux_df$year)

  efp_results_list[[this_site]] <- safeEFPcalc(flux_df)
}

Processing site: AT-Neu



[1] "#### Precipitation data available!!! Precipitation Filter Active"
Quality control:
TA: 2009 data points (0.6%) set to NA
H: 7462 data points (2.24%) set to NA
LE: 7385 data points (2.22%) set to NA
NEE: 25342 data points (7.61%) set to NA
VPD: 3472 data points (1.04%) set to NA
-------------------------------------------------------------------
Data filtering:
163872 data points (49.19%) excluded by growing season filter
72811 additional data points (21.86%) excluded by precipitation filter (124569
 data points = 37.39 % in total)
236683 data points (71.05%) excluded in total
96437 valid data points (28.95%) remaining.
[1] "....computing WUE Metrics for the site"
[1] "....computing maximum Evapotranspiration"


Warning message in EFPsOut$ETmax <- quantile(aaa$ET, 0.95, na.rm = T):
“Coercing LHS to a list”


[1] "....computing maximum surface conductance (Gsmax) and stomatal slope (G1)"
Note new column 'z0h' in the function output for 'bigleaf' versions > 0.7.5.
Energy storage fluxes S are not provided and set to 0.
Respiration from the leaves is ignored and set to 0.
[1] "....computing Evaporative fraction for the sites and its amplitude"
[1] "....computing LRC parameters for the site"
Quality control:
TA: 2009 data points (0.6%) set to NA
H: 7462 data points (2.24%) set to NA
LE: 7385 data points (2.22%) set to NA
NEE: 25342 data points (7.61%) set to NA
VPD: 3472 data points (1.04%) set to NA
-------------------------------------------------------------------
Data filtering:
163872 data points (49.19%) excluded by growing season filter
0 additional data points (0%) excluded by precipitation filter (0 data points = 
0 % in total)
163872 data points (49.19%) excluded in total
169248 valid data points (50.81%) remaining.
[1] "....computing Rb parameters for the site"
[1] "Calculation of Ti

Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message:
“Using 'fast' method. This can caus

...

Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message:
“Using 'fast' method. This can caus

[1] "#### Precipitation data available!!! Precipitation Filter Active"
Quality control:
TA: 314 data points (2.6%) set to NA
H: 9374 data points (77.75%) set to NA
LE: 9450 data points (78.38%) set to NA
NEE: 9540 data points (79.13%) set to NA
VPD: 328 data points (2.72%) set to NA
-------------------------------------------------------------------
Data filtering:
344 data points (2.85%) excluded by growing season filter
1494 additional data points (12.39%) excluded by precipitation filter (1608
 data points = 13.34 % in total)
1838 data points (15.25%) excluded in total
10218 valid data points (84.75%) remaining.
[1] "....computing WUE Metrics for the site"
[1] "....computing maximum Evapotranspiration"


Warning message in EFPsOut$ETmax <- quantile(aaa$ET, 0.95, na.rm = T):
“Coercing LHS to a list”


[1] "....computing maximum surface conductance (Gsmax) and stomatal slope (G1)"
Note new column 'z0h' in the function output for 'bigleaf' versions > 0.7.5.
Energy storage fluxes S are not provided and set to 0.
Respiration from the leaves is ignored and set to 0.
[1] "....computing Evaporative fraction for the sites and its amplitude"
[1] "....computing LRC parameters for the site"
Quality control:
TA: 314 data points (2.6%) set to NA
H: 9374 data points (77.75%) set to NA
LE: 9450 data points (78.38%) set to NA
NEE: 9540 data points (79.13%) set to NA
VPD: 328 data points (2.72%) set to NA
-------------------------------------------------------------------
Data filtering:
344 data points (2.85%) excluded by growing season filter
0 additional data points (0%) excluded by precipitation filter (0 data points = 
0 % in total)
344 data points (2.85%) excluded in total
11712 valid data points (97.15%) remaining.
[1] "....computing Rb parameters for the site"
[1] "Calculation of Time zone"


Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message in CPL_crs_from_input(x):
“GDAL Error 1: PROJ: proj_create_from_database: Open of /home/nk1125/miniconda3/envs/clean_r_env/share/proj failed”
Warning message:
“Using 'fast' method. This can caus

In [ ]:
# Initialize list to store results
efp_results_list <- list()

# Loop through each unique SITE_ID in tower_meta or Icos_metadata
for (this_site in unique(tower_meta$SITE_ID)) {
  message("Processing site: ", this_site)
  
  # Extract site info
  site_info <- tower_meta %>% filter(SITE_ID == this_site)
  
  # Skip if location info is missing
  if (any(is.na(site_info$LOCATION_ELEV), is.na(site_info$LOCATION_LAT), is.na(site_info$LOCATION_LONG))) {
    warning("Missing location info for site ", this_site, "; skipping.")
    next
  }
  
  # Convert site metadata
  site <- site_info$SITE_ID
  elevation <- as.numeric(site_info$LOCATION_ELEV)
  lat <- as.numeric(site_info$LOCATION_LAT)
  lon <- as.numeric(site_info$LOCATION_LONG)
  
  # Compute timezone
  TimeZone <- TimeZoneCalculatorFLUXNET(lat = lat, long = lon)
  
  # Subset flux data
  flux_df <- flux_all %>% filter(siteID == this_site)
  syear <- min(flux_df$year, na.rm = TRUE)
  eyear <- max(flux_df$year, na.rm = TRUE)
  nyears <- length(unique(flux_df$year))
  # Skip if no data
  if (nrow(flux_df) == 0) {
    warning("No data for site ", this_site, "; skipping.")
    next
  }
  
  # DateTime adjustments
  flux_df <- flux_df %>%
    mutate(
      ET = LE.to.ET(LE, TA) * 1800,
      precip_mm = P,
      VPD_kPa = VPD / 10,
      PPFD_IN_FROM_SWIN = SW_IN * 2.11,
      DateTime = DateTime + minutes(15)
    )
  
  # Run EFPcalc
  efp_result <- safeEFPcalc(flux_df)
  efp_results_list[[this_site]] <- efp_result
}

In [16]:
length(efp_results_list)
table(vapply(efp_results_list, function(x) class(x)[1], character(1)))
table(vapply(efp_results_list, is.data.frame, logical(1)))

[1] 469


logical 
    469 


FALSE 
  469 

In [11]:
library(purrr)
library(dplyr)


efp_results_list_df <- efp_results_list[sapply(efp_results_list, is.data.frame)]

efp_all <- bind_rows(efp_results_list_df)


In [12]:
colnames(efp_all)

character(0)

In [15]:
str(efp_all)

tibble [0 × 0] (S3: tbl_df/tbl/data.frame)
 Named list()


In [13]:
efp_with_meta <- efp_all %>%
  left_join(tower_meta, by = "SITE_ID")

ERROR: [1m[33mError[39m in `left_join()`:[22m
[1m[22m[33m![39m Join columns in `x` must be present in the data.
[31m✖[39m Problem with `SITE_ID`.


In [18]:
write.csv(efp_with_meta,"efp_per_site/EFP_per_sitesV0Dec.csv", row.names = FALSE)

In [20]:
# the sum of uique sites
length(unique(efp_with_meta$SITE_ID))

[1] 348

In [19]:
View(efp_with_meta)

SITE_ID,siteID,uWUE,ETmax,precipAvail,Gavail,GSmax,CO2avail,G1,EF,⋯,Rb,Rbmax,aCUE,TZ,nyears,SITE_NAME,LOCATION_LAT,LOCATION_LONG,LOCATION_ELEV,IGBP
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
AR-SLu,AR-SLu,14.6420243246375,0.119488149000771,yes,yes,0.00309720583049285,yes,-0.494621540398142,0.169642857142857,⋯,NA,NA,NA,-3,3,San Luis,-33.46480,-66.4598,505.00,MF
AR-Vir,AR-Vir,3.99903007043068,0.293510310932313,yes,yes,0.000729289456096901,yes,-0.828186739263869,0.7765636232108,⋯,NA,NA,NA,-3,4,Virasoro,-28.23950,-56.1886,105.00,ENF
AT-Neu,AT-Neu,4.74117812011236,0.27327688876603,yes,yes,0.0155729220960465,yes,1.4482808262471,0.822628331411729,⋯,15.3857008525882,21.7621260882325,0.69785387223731,1,19,Neustift,47.11670,11.3175,970.00,GRA
AU-Boy,AU-Boy,1.16433409547872,0.106848860628347,yes,yes,0.0176816893610645,yes,1.89778638152249,0.219260219944214,⋯,NA,NA,NA,8,6,,-32.47709,116.9386,390.00,EBF
AU-Emr,AU-Emr,2.68814667477306,0.126297222352692,yes,yes,0.00282344764463533,yes,1.48334056440369,0.292237442922374,⋯,NA,NA,NA,10,3,Emerald,-23.85870,148.4746,179.00,GRA
AU-Gin,AU-Gin,3.20488022956418,0.137291869191616,yes,yes,0.00428733052811933,yes,2.2375730535591,0.301003344481605,⋯,NA,NA,NA,8,4,Gingin,-31.37640,115.7138,51.00,WSA
AU-GWW,AU-GWW,1.03617142742862,0.0785469801040957,yes,yes,0.00789202210493344,yes,1.62663968457448,0.142857142857143,⋯,NA,NA,NA,8,10,"Great Western Woodlands, Western Australia, Australia",-30.19130,120.6541,453.00,SAV
AU-Lon,AU-Lon,1.06875925442685,0.18076941626636,yes,yes,0.016106208415338,yes,2.00097220742241,0.496093511566868,⋯,NA,NA,NA,10,5,,-23.52327,144.3104,207.00,GRA
AU-Rgf,AU-Rgf,0.62296456956519,0.16411283710531,yes,yes,0.0262309129232667,yes,2.03379124058347,0.580917366946779,⋯,NA,NA,NA,8,8,,-32.50610,116.9668,311.00,CRO


In [ ]:
library(data.table)
library(dplyr)
library(purrr)
library(lubridate)

tower_meta <- fread("clean_data/combined_site_metadata.csv")
flux_all <- "merged_sites_subset_renamed"

In [ ]:
safeEFPcalc <- purrr::possibly(EFPcalc, otherwise = NA)

In [ ]:
# Initialize results list
efp_results_list <- list()

# Loop over each site
for (this_site in unique(tower_meta$SITE_ID)) {
  message("Processing site: ", this_site)
  
  # Extract metadata
  site_info <- tower_meta %>% filter(SITE_ID == this_site)
  
  # Skip if any location info is missing
  if (any(is.na(site_info$LOCATION_ELEV), is.na(site_info$LOCATION_LAT), is.na(site_info$LOCATION_LONG))) {
    warning("Missing location info for site ", this_site, "; skipping.")
    next
  }
  
  lat <- as.numeric(site_info$LOCATION_LAT)
  lon <- as.numeric(site_info$LOCATION_LONG)
  elevation <- as.numeric(site_info$LOCATION_ELEV)
  
  # Compute timezone
  TimeZone <- TimeZoneCalculatorFLUXNET(lat = lat, long = lon)
  
  # Subset flux data
  flux_df <- flux_all %>% filter(siteID == this_site)
  
  # Skip if no data
  if (nrow(flux_df) == 0) {
    warning("No data for site ", this_site, "; skipping.")
    next
  }
  
  # Adjust variables
  flux_df <- flux_df %>%
    mutate(
      ET = LE.to.ET(LE, TA) * 1800,
      precip_mm = P,
      VPD_kPa = VPD / 10,
      PPFD_IN_FROM_SWIN = SW_IN * 2.11,
      DateTime = DateTime + minutes(15)
    )
  
  # Loop over each year for this site
  for (yr in unique(flux_df$year)) {
    message("  → Calculating EFPs for year: ", yr)
    
    dat_year <- filter(flux_df, year == yr)
    
    efp_result <- safeEFPcalc(dat_year)
    
    if (!is.null(efp_result) && is.data.frame(efp_result)) {
      efp_result$SITE_ID <- this_site
      efp_result$year <- yr
      efp_results_list[[paste0(this_site, "_", yr)]] <- efp_result
    }
  }
}


In [ ]:
# Combine all valid results
efp_results_list_df <- efp_results_list[sapply(efp_results_list, is.data.frame)]
efp_all <- bind_rows(efp_results_list_df)

# Join with metadata
efp_with_meta <- left_join(efp_all, tower_meta, by = "SITE_ID")


In [ ]:
# Save to file
write.csv(efp_with_meta, "clean_data/efp_by_site_year_V2.csv", row.names = FALSE)
